# Week 5 — XAI Baseline Explanations

**Goal:** Generate GradCAM explanations for DenseNet-121 and Attention Rollout maps for ViT-B/16 on the same images, then produce a side-by-side comparison figure.

**Deliverables:**
- GradCAM heatmaps overlaid on chest X-ray images (DenseNet-121)
- Attention Rollout maps overlaid on the same images (ViT-B/16)
- Side-by-side comparison figure saved to `figures/`
- Initial observation: do the two models highlight similar or different regions?

Run on **Google Colab** with T4 GPU.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torchxrayvision transformers timm grad-cam scikit-image seaborn

In [ ]:
import numpy as np
import torch
import torchxrayvision as xrv
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from transformers import ViTForImageClassification, ViTImageProcessor

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {device}")

Path("../figures").mkdir(exist_ok=True)

## 1. Load Both Models

In [ ]:
# DenseNet-121 — CheXpert pretrained
densenet = xrv.models.DenseNet(weights="densenet121-res224-chex").eval().to(device)

# ViT-B/16
PATHOLOGIES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
VIT_CHECKPOINT = "google/vit-base-patch16-224-in21k"
vit_processor = ViTImageProcessor.from_pretrained(VIT_CHECKPOINT)
vit_model = ViTForImageClassification.from_pretrained(
    VIT_CHECKPOINT,
    num_labels=len(PATHOLOGIES),
    ignore_mismatched_sizes=True,
).eval().to(device)

print("Both models loaded.")

## 2. Load Sample Images

Swap `get_sample_images()` for `load_chexpert_samples()` once CheXpert is approved.

In [ ]:
def get_sample_images(n: int = 5):
    """Returns list of (xrv_arr, pil_rgb) tuples."""
    raw = xrv.datasets.utils.get_sample_data()['PA']             # (1, H, W) float
    raw = xrv.datasets.XRayCenterCrop()(raw)
    raw = xrv.datasets.XRayResizer(224)(raw)                     # (1, 224, 224)
    xrv_arr = xrv.datasets.utils.normalize(raw, maxval=255, reshape=True)  # (1,224,224)

    # Convert to uint8 RGB for ViT processor and visualisation
    uint8 = ((xrv_arr[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    pil_rgb = Image.fromarray(uint8).convert("RGB")

    return [(xrv_arr.copy(), pil_rgb.copy()) for _ in range(n)]


def load_chexpert_samples(folder: str, n: int = 5):
    """Load n images from a CheXpert folder once approved."""
    paths = sorted(Path(folder).rglob("*.jpg"))[:n]
    results = []
    for p in paths:
        img = Image.open(p).convert("L").resize((224, 224))
        arr = np.array(img, dtype=np.float32)
        xrv_arr = xrv.datasets.utils.normalize(arr, maxval=255, reshape=True)
        uint8 = ((xrv_arr[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
        pil_rgb = Image.fromarray(uint8).convert("RGB")
        results.append((xrv_arr, pil_rgb))
    return results


# images = load_chexpert_samples("../data/CheXpert-v1.0-small/train", n=5)
images = get_sample_images(n=5)
print(f"Loaded {len(images)} images")

## 3. GradCAM for DenseNet-121

Target layer: the last conv layer of `denseblock4` — this is the deepest feature map before the classifier, giving the most semantically rich activations.

In [ ]:
# Inspect DenseNet feature block names (run once to verify)
for name, _ in densenet.features.denseblock4.named_children():
    pass  # last `name` printed below
print(f"Last layer in denseblock4: {name}")

# Target the last conv layer in the last denselayer
DENSENET_TARGET_LAYER = densenet.features.denseblock4[-1].conv2

In [ ]:
def get_gradcam(xrv_arr: np.ndarray, target_class: int, use_plusplus: bool = False) -> np.ndarray:
    """
    Generate a GradCAM heatmap (H, W) in [0,1] for DenseNet-121.

    Args:
        xrv_arr:      (1, 224, 224) float array normalised for torchxrayvision
        target_class: index into densenet.pathologies
        use_plusplus: use GradCAM++ instead of GradCAM
    """
    tensor = torch.from_numpy(xrv_arr).unsqueeze(0).to(device)  # (1,1,224,224)
    cam_cls = GradCAMPlusPlus if use_plusplus else GradCAM
    with cam_cls(model=densenet, target_layers=[DENSENET_TARGET_LAYER]) as cam:
        targets = [ClassifierOutputTarget(target_class)]
        heatmap = cam(input_tensor=tensor, targets=targets)[0]   # (224, 224)
    return heatmap


# Pick the class index for "Cardiomegaly" as a demo
TARGET_CLASS_DENSENET = densenet.pathologies.index("Cardiomegaly")
print(f"Target class: {TARGET_CLASS_DENSENET} — {densenet.pathologies[TARGET_CLASS_DENSENET]}")

gradcam_maps = [get_gradcam(xrv_arr, TARGET_CLASS_DENSENET) for xrv_arr, _ in images]
print(f"Generated {len(gradcam_maps)} GradCAM maps  |  shape: {gradcam_maps[0].shape}")

## 4. Attention Rollout for ViT-B/16

Attention Rollout (Abnar & Zuidema, 2020) propagates attention through all transformer layers by multiplying the attention matrices with skip connections. The result is a 14×14 map (one value per patch) showing which image regions the model attends to globally.

In [ ]:
def attention_rollout(pil_rgb: Image.Image, discard_ratio: float = 0.9) -> np.ndarray:
    """
    Compute Attention Rollout for ViT-B/16.

    Returns a (224, 224) float array in [0,1] — upsampled from the 14×14 patch grid.

    Args:
        pil_rgb:       RGB PIL image (224×224)
        discard_ratio: fraction of lowest-attention tokens zeroed out per layer
    """
    inputs = vit_processor(images=pil_rgb, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = vit_model(**inputs, output_attentions=True)

    # attentions: tuple of (1, num_heads, seq_len, seq_len) per layer
    # seq_len = 197 = 196 patch tokens + 1 CLS token
    attentions = outputs.attentions

    result = torch.eye(attentions[0].size(-1)).to(device)  # (197, 197)

    for attn in attentions:
        # Average over attention heads → (1, 197, 197)
        attn_avg = attn.mean(dim=1).squeeze(0)             # (197, 197)

        # Zero out the lowest-attention entries (discard ratio)
        flat = attn_avg.flatten()
        threshold = torch.quantile(flat, discard_ratio)
        attn_avg = torch.where(attn_avg < threshold, torch.zeros_like(attn_avg), attn_avg)

        # Add residual / skip connection, then re-normalise rows
        attn_avg = attn_avg + torch.eye(attn_avg.size(0), device=device)
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)

        result = attn_avg @ result

    # Row 0 = CLS token attending to all patches; columns 1: = patch tokens
    mask = result[0, 1:].cpu().numpy()    # (196,)
    side = int(np.sqrt(mask.size))        # 14
    mask = mask.reshape(side, side)       # (14, 14)

    # Upsample to 224×224
    mask_img = Image.fromarray((mask / mask.max() * 255).astype(np.uint8)).resize(
        (224, 224), Image.BILINEAR
    )
    return np.array(mask_img, dtype=np.float32) / 255.0


rollout_maps = [attention_rollout(pil_rgb) for _, pil_rgb in images]
print(f"Generated {len(rollout_maps)} Attention Rollout maps  |  shape: {rollout_maps[0].shape}")

## 5. Helper — Overlay Heatmap on Image

In [ ]:
def overlay(pil_rgb: Image.Image, heatmap: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Blend a (H,W) float heatmap over an RGB PIL image. Returns (H,W,3) uint8."""
    img_arr = np.array(pil_rgb, dtype=np.float32)
    heat_rgb = (cm.jet(heatmap)[:, :, :3] * 255).astype(np.float32)
    return (alpha * heat_rgb + (1 - alpha) * img_arr).clip(0, 255).astype(np.uint8)

## 6. Side-by-Side Comparison Figure

Each row = one image. Columns: original | GradCAM (DenseNet) | Attention Rollout (ViT)

In [ ]:
n = len(images)
fig, axes = plt.subplots(n, 3, figsize=(10, 3.5 * n))
fig.suptitle(
    f"XAI Explanations: DenseNet-121 (GradCAM) vs ViT-B/16 (Attention Rollout)\n"
    f"Target: {densenet.pathologies[TARGET_CLASS_DENSENET]}",
    fontsize=13, y=1.01
)

col_titles = ["Original X-ray", "GradCAM (DenseNet-121)", "Attention Rollout (ViT-B/16)"]

for i, ((xrv_arr, pil_rgb), gc_map, ar_map) in enumerate(zip(images, gradcam_maps, rollout_maps)):
    row = axes[i] if n > 1 else axes

    # Original
    row[0].imshow(np.array(pil_rgb), cmap="gray")
    row[0].axis("off")
    if i == 0:
        row[0].set_title(col_titles[0], fontsize=10, fontweight="bold")

    # GradCAM overlay
    row[1].imshow(overlay(pil_rgb, gc_map))
    row[1].axis("off")
    if i == 0:
        row[1].set_title(col_titles[1], fontsize=10, fontweight="bold")

    # Attention Rollout overlay
    row[2].imshow(overlay(pil_rgb, ar_map))
    row[2].axis("off")
    if i == 0:
        row[2].set_title(col_titles[2], fontsize=10, fontweight="bold")

    row[0].set_ylabel(f"Image {i+1}", fontsize=9)

plt.tight_layout()
out_path = "../figures/week5_xai_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

## 7. Per-image SSIM between GradCAM and Attention Rollout

A quick look at how structurally different the two explanation maps are — before any perturbations.

In [ ]:
from skimage.metrics import structural_similarity as ssim

print("SSIM between GradCAM and Attention Rollout (same image, no perturbation)\n" + "-"*55)
for i, (gc, ar) in enumerate(zip(gradcam_maps, rollout_maps)):
    score = ssim(gc, ar, data_range=1.0)
    print(f"  Image {i+1}: SSIM = {score:.4f}")

## 8. Initial Observations

Write your answers here after running the notebook:

1. **Spatial focus** — Does GradCAM produce sharp, localised activations or diffuse ones? What about Attention Rollout?
2. **Region agreement** — Do both models highlight similar anatomical regions for the same image?
3. **SSIM scores** — Are the two explanation maps structurally similar (SSIM > 0.7) or very different?

*These observations will form the starting point of your Discussion section.*